In [ ]:
!pip install -U ranx
import ranx


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.0/859.0 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.0/135.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 2.2 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=71df99138505b63b75f03357e92af849b116e3e291b6a3ad916f26bb2b124b16
  Stored in directory: /root/.cache/pip/wheels/63/f9/dc/2dd16d3330e327236e4d407941975c42d5159d200cdb7922d8
  Created wheel for cbor: filename=cbor-1.0.0-cp311-cp311-linux_x86_64.whl size=539

In [ ]:
#Loading the ranking.run files generated after the evaluation
from google.colab import drive
drive.mount('/content/gdrive')

# Load the run files
run1 = ranx.Run.from_file(
    "./gdrive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-distilroberta-base-2025-05-19_08-04-18/ranking.run",
    kind="trec"
)

run2 = ranx.Run.from_file(
    "./gdrive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2025-05-18_15-00-51/ranking.run",
    kind="trec"
)

run3 = ranx.Run.from_file(
    "./gdrive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-MiniLM-L-2-v2-2025-05-24_13-14-18/ranking.run",
    kind="trec"
)


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
# Loading the qrels file for evaluation by downloading
!wget https://trec.nist.gov/data/deep/2019qrels-pass.txt -O 2019qrels-pass.txt

# Now load it using Ranx
qrels = ranx.Qrels.from_file("2019qrels-pass.txt", kind="trec")

--2025-05-24 15:13:07--  https://trec.nist.gov/data/deep/2019qrels-pass.txt
Resolving trec.nist.gov (trec.nist.gov)... 172.65.90.27, 172.65.90.26, 172.65.90.24, ...
Connecting to trec.nist.gov (trec.nist.gov)|172.65.90.27|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 187092 (183K) [text/plain]
Saving to: ‘2019qrels-pass.txt’

2019qrels-pass.txt  100%[===================>] 182.71K  --.-KB/s    in 0.08s   

2025-05-24 15:13:08 (2.28 MB/s) - ‘2019qrels-pass.txt’ saved [187092/187092]



In [ ]:
import ranx

# Print out available fusion methods
print(ranx.fusion.__all__)

['bayesfuse', 'bayesfuse_train', 'bordafuse', 'comb_anz', 'comb_gmnz', 'comb_max', 'comb_med', 'comb_min', 'comb_mnz', 'comb_sum', 'condorcet', 'isr', 'log_isr', 'logn_isr', 'mapfuse_train', 'mapfuse', 'mixed', 'posfuse_train', 'posfuse', 'probfuse_train', 'probfuse', 'rbc', 'rrf', 'segfuse_train', 'segfuse', 'slidefuse_train', 'slidefuse', 'weighted_bordafuse', 'weighted_condorcet', 'wmnz', 'wsum']


In [ ]:
#Applying five methods that are selected from the ranx which are log_isr,reciprocal rank fusion, wsum,comb_sum and condorcet_fuse
from ranx import fuse, evaluate
# Apply log_isr ensemble method
log_isr_ensemble = ranx.fuse([run1, run2, run3], method='log_isr')
log_isr_result = evaluate(qrels, log_isr_ensemble, metrics=["ndcg@10", "recall@100", "map@1000"])

# Apply reciprocal rank fusion ensemble method
rrf_ensemble = ranx.fuse([run1, run2, run3], method='rrf')
rrf_result = evaluate(qrels, rrf_ensemble, metrics=["ndcg@10", "recall@100", "map@1000"])


# Apply weighted_sum ensemble method
optimal_weights = [0.4, 0.3, 0.3]  # Assume these were found to optimize MAP
wsum_ensemble = ranx.fuse([run1, run2, run3], method="wsum", params={"weights": optimal_weights})
wsum_result = evaluate(qrels, wsum_ensemble, metrics=["ndcg@10", "recall@100", "map@1000"])

#Applying mapfuse method
map_scores = []
for run in [run1, run2, run3]:
    result = evaluate(qrels, run, metrics=["map"])
    map_scores.append(result)  # 'result' will be a scalar value representing the MAP score

# Apply the mapfuse ensemble method with the computed map_scores
mapfuse_ensemble = fuse([run1, run2, run3], method="mapfuse", params={"map_scores": map_scores})
mapfuse_result = evaluate(qrels, mapfuse_ensemble, metrics=["ndcg@10", "recall@100", "map@1000"])


# Apply condorcet_fuse ensemble method
condorcet_ensemble = ranx.fuse([run1, run2, run3], method='condorcet')
condorcet_result = evaluate(qrels, condorcet_ensemble, metrics=["ndcg@10", "recall@100", "map@1000"])



In [ ]:
# Print the evaluation results for each ensemble method
import numpy as np

def convert_to_float(value):
    return float(value) if isinstance(value, np.float64) else value

print("log_ISR Evaluation:")
print({key: convert_to_float(val) for key, val in log_isr_result.items()})

print("\nReciprocal Rank Fusion Evaluation:")
print({key: convert_to_float(val) for key, val in rrf_result.items()})

print("\nWeighted Sum Evaluation:")
print({key: convert_to_float(val) for key, val in wsum_result.items()})

print("\nMAPFuse Evaluation:")
print({key: convert_to_float(val) for key, val in mapfuse_result.items()})

print("\nCondorcet Evaluation:")
print({key: convert_to_float(val) for key, val in condorcet_result.items()})

log_ISR Evaluation:
{'ndcg@10': 0.6708542308447024, 'recall@100': 0.5058668072976452, 'map@1000': 0.4450919416965035}

Reciprocal Rank Fusion Evaluation:
{'ndcg@10': 0.7006962807764576, 'recall@100': 0.5123601735277414, 'map@1000': 0.4604264371664257}

Weighted Sum Evaluation:
{'ndcg@10': 0.689267983761608, 'recall@100': 0.5109144923516071, 'map@1000': 0.45300884259708235}

MAPFuse Evaluation:
{'ndcg@10': 0.6805123411861793, 'recall@100': 0.5083481698858829, 'map@1000': 0.45119526371651286}

Condorcet Evaluation:
{'ndcg@10': 0.6937266824242629, 'recall@100': 0.5058732061045675, 'map@1000': 0.457371326036382}


In [ ]:
import pandas as pd

# Task 2 Results - make sure these variable names match your actual result objects
results = {
    'Method': ['log_isr', 'rrf', 'weighted_sum', 'map_fuse', 'condorcet_fuse'],
    'NDCG@10': [
        log_isr_result['ndcg@10'],
        rrf_result['ndcg@10'],
        wsum_result['ndcg@10'],
        mapfuse_result['ndcg@10'],
        condorcet_result['ndcg@10'],
    ],
    'Recall@100': [
        log_isr_result['recall@100'],
        rrf_result['recall@100'],
        wsum_result['recall@100'],
        mapfuse_result['recall@100'],
        condorcet_result['recall@100'],
    ],
    'MAP@1000': [
        log_isr_result['map@1000'],
        rrf_result['map@1000'],
        wsum_result['map@1000'],
        mapfuse_result['map@1000'],
        condorcet_result['map@1000'],
    ]
}

# Create DataFrame
df = pd.DataFrame(results)

# Print in notebook
print(df)

# Optional: Save for report use
#df.to_csv("task2_ensemble_results.csv", index=False)  # For Word/Excel
#print(df.to_latex(index=False, float_format="%.4f"))  # For Overleaf or LaTeX reports


           Method   NDCG@10  Recall@100  MAP@1000
0         log_isr  0.670854    0.505867  0.445092
1             rrf  0.700696    0.512360  0.460426
2    weighted_sum  0.689268    0.510914  0.453009
3        map_fuse  0.680512    0.508348  0.451195
4  condorcet_fuse  0.693727    0.505873  0.457371


**Task 3 considering the reciprocal rank fusion and fusing the runs to evaluate the combined results**

In [ ]:
# Fusion with RRF for each pair
rrf_1_2 = ranx.fuse([run1, run2], method="rrf")
rrf_1_3 = ranx.fuse([run1, run3], method="rrf")
rrf_2_3 = ranx.fuse([run2, run3], method="rrf")

# Evaluate all combinations
rrf_1_2_result = evaluate(qrels, rrf_1_2, metrics=["ndcg@10", "recall@100", "map@1000"])
rrf_1_3_result = evaluate(qrels, rrf_1_3, metrics=["ndcg@10", "recall@100", "map@1000"])
rrf_2_3_result = evaluate(qrels, rrf_2_3, metrics=["ndcg@10", "recall@100", "map@1000"])

# Print results
print("distilroberta + tinybert RRF score:")
print(f"  NDCG@10: {rrf_1_2_result['ndcg@10']:.4f}")
print(f"  Recall@100: {rrf_1_2_result['recall@100']:.4f}")
print(f"  MAP@1000: {rrf_1_2_result['map@1000']:.4f}\n")

print("distilroberta + minilm RRF score:")
print(f"  NDCG@10: {rrf_1_3_result['ndcg@10']:.4f}")
print(f"  Recall@100: {rrf_1_3_result['recall@100']:.4f}")
print(f"  MAP@1000: {rrf_1_3_result['map@1000']:.4f}\n")

print("tinybert + minilm RRF score:")
print(f"  NDCG@10: {rrf_2_3_result['ndcg@10']:.4f}")
print(f"  Recall@100: {rrf_2_3_result['recall@100']:.4f}")
print(f"  MAP@1000: {rrf_2_3_result['map@1000']:.4f}")




distilroberta + tinybert RRF score:
  NDCG@10: 0.7003
  Recall@100: 0.5046
  MAP@1000: 0.4590

distilroberta + minilm RRF score:
  NDCG@10: 0.6909
  Recall@100: 0.4999
  MAP@1000: 0.4431

tinybert + minilm RRF score:
  NDCG@10: 0.6954
  Recall@100: 0.5066
  MAP@1000: 0.4559


In [ ]:
#Maintaining in the tabular format of the scores and also not limiting to 4 decimal places
import pandas as pd

t3_results = {
    'Combination': [
        'distilroberta + tinybert',
        'distilroberta + minilm',
        'tinybert + minilm'
    ],
    'nDCG@10': [
        rrf_1_2_result['ndcg@10'],
        rrf_1_3_result['ndcg@10'],
        rrf_2_3_result['ndcg@10']
    ],
    'Recall@100': [
        rrf_1_2_result['recall@100'],
        rrf_1_3_result['recall@100'],
        rrf_2_3_result['recall@100']
    ],
    'MAP@1000': [
        rrf_1_2_result['map@1000'],
        rrf_1_3_result['map@1000'],
        rrf_2_3_result['map@1000']
    ]
}

task3_df = pd.DataFrame(t3_results)
print("\nTask 3 Pairwise RRF Results:")
print(task3_df)




Task 3 Pairwise RRF Results:
                Combination   nDCG@10  Recall@100  MAP@1000
0  distilroberta + tinybert  0.700262    0.504565  0.458974
1    distilroberta + minilm  0.690926    0.499892  0.443096
2         tinybert + minilm  0.695432    0.506562  0.455927
